[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_07_research_capstone/ch36_flagship_research_capstone/03_coding_exam.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 36 Grand Book Finale Capstone Exam — From Raw Survey CSV to Publication-Ready Release (100 Points)

**Book Reference:** Chapter 36, Section 36.7 (Grand Book Finale Capstone Exam)  
**Suggested Time:** 60 minutes (Open Book, Closed Solution)

| Task | Flagship Research Capstone Deliverable | Points |
| :--- | :--- | :---: |
| **Task 36.1** | `full_table1_and_adjusted_or(mch_df)` (Journal Table 1 + multivariable `statsmodels.Logit` aORs) | 30 |
| **Task 36.2** | `capstone_ml_dl_fairness_pipeline(mch_df, seed=42)` (5-fold `GroupKFold` LR/HGB/PyTorch MLP + Youden fairness + DCA) | 35 |
| **Task 36.3** | `execute_first_paper_release(mch_df, release_dir, seed=42)` (Full 7-artifact release + `joblib` Pipeline + SHA-256 manifest) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Grand Finale Capstone Grader
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import sharukh_tutor as tutor

torch.set_num_threads(1)
WEALTH_MAP = {
    "Poorest": 1, "Poorer": 2, "Middle": 3,
    "Richer": 4, "Richest": 5,
}
FEATURES = [
    "maternal_age_years", "anc_visits",
    "maternal_hb_g_dl", "is_urban", "wealth_rank",
]


def clean_mch_survey(mch_df):
    """Provided helper: clean sentinels, residence casing & wealth."""
    df = mch_df.copy()
    anc = pd.to_numeric(df["anc_visits"], errors="coerce")
    df["anc_visits"] = anc.replace(999, np.nan)
    df["residence"] = (
        df["residence"].astype(str).str.strip().str.title()
    )
    df["is_urban"] = (df["residence"] == "Urban").astype(int)
    df["wealth_rank"] = df["wealth_quintile"].map(WEALTH_MAP)
    df = df.dropna(subset=["child_stunted"]).copy()
    df["child_stunted"] = df["child_stunted"].astype(int)
    return df


def build_publication_table1(mch_df):
    """Provided Chapter 36 helper (To-Do 36.1)."""
    df = clean_mch_survey(mch_df)
    g0 = df[df["child_stunted"] == 0]
    g1 = df[df["child_stunted"] == 1]
    rows = []
    for col in ["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]:
        s0, s1 = g0[col].dropna(), g1[col].dropna()
        _, p_val = stats.ttest_ind(s0, s1, equal_var=False)
        rows.append({
            "variable": col,
            "not_stunted": f"{s0.mean():.2f} ± {s0.std(ddof=1):.2f}",
            "stunted": f"{s1.mean():.2f} ± {s1.std(ddof=1):.2f}",
            "p_value": round(float(p_val), 4),
        })
    ct = pd.crosstab(df["child_stunted"], df["is_urban"])
    _, p_chi, _, _ = stats.chi2_contingency(ct, correction=False)
    c0, pct0 = int(g0["is_urban"].sum()), 100.0 * g0["is_urban"].mean()
    c1, pct1 = int(g1["is_urban"].sum()), 100.0 * g1["is_urban"].mean()
    rows.append({
        "variable": "is_urban",
        "not_stunted": f"{c0} ({pct0:.1f}%)",
        "stunted": f"{c1} ({pct1:.1f}%)",
        "p_value": round(float(p_chi), 4),
    })
    return pd.DataFrame(rows)


def compute_clinical_net_benefit(y_true, y_prob, thresholds):
    """Provided Chapter 36 helper (To-Do 36.4)."""
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_prob, dtype=float)
    n, prev = len(yt), float(np.mean(yt))
    rows = []
    for t in thresholds:
        t_f = float(t)
        if not (0.0 < t_f < 1.0):
            raise ValueError(f"Threshold {t_f} must satisfy 0 < t < 1")
        pred = (yp >= t_f).astype(int)
        tp = int(np.sum((pred == 1) & (yt == 1)))
        fp = int(np.sum((pred == 1) & (yt == 0)))
        odds = t_f / (1.0 - t_f)
        rows.append({
            "threshold": round(t_f, 5),
            "nb_model": round((tp / n) - (fp / n) * odds, 5) + 0.0,
            "nb_all": round(prev - (1.0 - prev) * odds, 5) + 0.0,
            "nb_none": 0.0,
        })
    return pd.DataFrame(rows)


def audit_manuscript_reproducibility(release_dir):
    """Provided Chapter 36 helper (To-Do 36.6)."""
    required = sorted([
        "MODEL_CARD.md", "fairness_audit.json", "leaderboard.csv",
        "manifest.json", "model.joblib", "table1.csv",
    ])
    root = Path(release_dir)
    present = sorted(f for f in required if (root / f).is_file())
    missing = sorted(f for f in required if not (root / f).is_file())
    return {
        "present": present,
        "missing": missing,
        "ready_for_submission": len(missing) == 0,
    }


tutor.activate(chapter="ch36", notebook="coding_exam")


---
## Exam Task 36.1 (30 Points): `full_table1_and_adjusted_or(mch_df)`

Build Table 1 and multivariable adjusted Odds Ratios:
1. Compute `table1 = build_publication_table1(mch_df)`.
2. Clean `mch_df` and select complete cases across `["child_stunted"] + FEATURES`.
3. Fit `statsmodels.api.Logit` predicting `child_stunted` from `sm.add_constant(X)`.
4. Build `adjusted_ors` DataFrame `["feature", "aor", "ci_low", "ci_high", "p_value"]` (excluding `'const'`, rounded to 4 decimals).
5. Return `{"table1": DataFrame, "adjusted_ors": DataFrame, "n_analytic": int}`.

In [ ]:
def full_table1_and_adjusted_or(mch_df):
    """Produce Journal Table 1 and multivariable adjusted ORs."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace with your solution")


---
## Exam Task 36.2 (35 Points): `capstone_ml_dl_fairness_pipeline(mch_df, seed=42)`

Run an end-to-end 5-fold `GroupKFold` (`cluster_id`) ML/DL benchmark, fairness audit, and DCA:
1. Benchmark `LogisticRegression`, `HistGradientBoostingClassifier`, and a 16-unit PyTorch `MLP` (`Linear -> ReLU -> Linear`, 30 epochs, train-only median imputation & `StandardScaler` inside each fold), storing out-of-fold probabilities.
2. Select `winner = leaderboard.iloc[0]["model"]` (sorted by `mean_auc` desc).
3. On the winner's OOF probabilities, find Youden's $J$ threshold (`argmax(tpr - fpr)`) and compute Rural vs. Urban fairness gaps (`threshold`, `demographic_parity_diff`, `equal_opportunity_diff`).
4. Compute Decision Curve Analysis (`dca_table`) across `[0.10, 0.15, 0.20, 0.25, 0.30]`.
5. Return `{"leaderboard": DataFrame, "winner": str, "fairness_gaps": dict, "dca_table": DataFrame}`.

In [ ]:
def capstone_ml_dl_fairness_pipeline(mch_df, seed=42):
    """Benchmark LR/HGB/PyTorch MLP + OOF fairness + DCA."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace with your solution")


---
## Exam Task 36.3 (35 Points): `execute_first_paper_release(mch_df, release_dir, seed=42)`

Execute the complete flagship release into `release_dir`:
1. Run `build_publication_table1(mch_df)` and `capstone_ml_dl_fairness_pipeline(mch_df, seed=seed)`.
2. Fit a standard sklearn `Pipeline` (`SimpleImputer` + `StandardScaler` + `LogisticRegression`) on the cleaned features and serialize it to `release_dir / "model.joblib"` via `joblib.dump`.
3. Compute the exact SHA-256 hex digest of `model.joblib`.
4. Write all 7 artifacts (`table1.csv`, `leaderboard.csv`, `fairness_audit.json`, `MODEL_CARD.md`, `model.joblib`, `config.json`, `manifest.json` embedding the SHA-256) into `release_dir`.
5. Audit `release_dir` with `audit_manuscript_reproducibility(release_dir)` and return `{"ready_for_submission": bool, "winner": str, "winner_auc": round4, "artifact_sha256": str}`.

In [ ]:
def execute_first_paper_release(mch_df, release_dir, seed=42):
    """Package and verify a complete publication release bundle."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace with your solution")


In [ ]:
exam_submissions = {
    "ch36_exam_01": full_table1_and_adjusted_or,
    "ch36_exam_02": capstone_ml_dl_fairness_pipeline,
    "ch36_exam_03": execute_first_paper_release,
}

tutor.check_exam("ch36", exam_submissions)
